# Step 2 — Government Quality Score (MIPS)

**Business question:** Given a list of Massachusetts provider NPIs, what is each provider's CMS MIPS quality score?

**Logic:** Pull QPP/MIPS final scores, enrich with Care Compare and Part B utilization data, and produce a clean per-NPI table. Business logic (three-situation classification) will be designed after exploring this data.

**Structural ceiling:** All CMS data is Medicare fee-for-service. Providers with mostly commercial patients will look thin in this data. This is a known limitation, not something we can engineer around.

## Step 1 — Load Data

**What this does:** Load the four datasets needed for the MIPS quality score pipeline:
1. **QPP/MIPS Final Scores** — CMS's official quality assessment per provider
2. **Care Compare** — clinician-level enrichment (telehealth, affiliations)
3. **Part B Utilization** — billing volume and specialty data
4. **NPPES** — provider registry filtered to Massachusetts (our master NPI list)

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import os
import glob

# --- File detection ---
# QPP/MIPS: try known filename patterns
qpp_candidates = glob.glob("2_datasets/ec_score*.csv") + glob.glob("2_datasets/MIPS*2023*.csv") + glob.glob("2_datasets/*qpp*.csv") + glob.glob("2_datasets/*QPP*.csv")
if not qpp_candidates:
    raise FileNotFoundError(
        "QPP/MIPS file not found. Expected 'ec_score_file.csv' or similar in 2_datasets/. "
        "Download from https://data.cms.gov/provider-data/topics/doctors-clinicians (QPP section)"
    )
qpp_file = qpp_candidates[0]
print(f"Using QPP file: {qpp_file}")

In [ ]:
# Load QPP/MIPS final scores
qpp = pd.read_csv(qpp_file, dtype=str)
print(f"QPP/MIPS: {qpp.shape[0]:,} rows, {qpp.shape[1]} columns")
print(f"Columns: {list(qpp.columns)}")

# Sanity check: expect ~800k-1M rows nationally
if qpp.shape[0] < 500_000:
    print(f"⚠️ WARNING: Only {qpp.shape[0]:,} rows. Expected ~800k-1M. Check if this is the right file/year.")
elif qpp.shape[0] > 2_000_000:
    print(f"⚠️ WARNING: {qpp.shape[0]:,} rows is unusually high. May contain multiple years.")
else:
    print(f"✓ Row count looks reasonable for national MIPS data")

In [ ]:
# Load Care Compare clinician data
cc_candidates = glob.glob("2_datasets/DAC_National*.csv") + glob.glob("2_datasets/*care_compare*.csv")
if not cc_candidates:
    raise FileNotFoundError(
        "Care Compare file not found. Expected 'DAC_NationalDownloadableFile.csv' in 2_datasets/. "
        "Download from https://data.cms.gov/provider-data/topics/doctors-clinicians"
    )
cc_file = cc_candidates[0]
print(f"Using Care Compare file: {cc_file}")

care_compare = pd.read_csv(cc_file, dtype=str)
print(f"Care Compare: {care_compare.shape[0]:,} rows, {care_compare.shape[1]} columns")
print(f"Columns: {list(care_compare.columns)}")

In [ ]:
# Load Medicare Part B utilization
partb_candidates = (
    glob.glob("2_datasets/MUP_PHY*Prov_Svc*.csv")
    + glob.glob("2_datasets/Medicare_Physician*Provider_and_Service*.csv")
)
if not partb_candidates:
    raise FileNotFoundError(
        "Part B file not found. Expected 'MUP_PHY_R25_P05_V20_D23_Prov_Svc.csv' in 2_datasets/. "
        "Download from https://data.cms.gov/provider-summary-by-type-of-service/"
    )
partb_file = partb_candidates[0]
print(f"Using Part B file: {partb_file}")
print(f"File size: {os.path.getsize(partb_file) / 1e9:.1f} GB")

# Load with only needed columns to save memory (~3GB file)
partb_usecols = [
    "Rndrng_NPI", "Rndrng_Prvdr_Last_Org_Name", "Rndrng_Prvdr_First_Name",
    "Rndrng_Prvdr_Type", "Rndrng_Prvdr_State_Abrvtn",
    "HCPCS_Cd", "HCPCS_Desc", "Place_Of_Srvc",
    "Tot_Benes", "Tot_Srvcs", "Tot_Bene_Day_Srvcs",
    "Avg_Sbmtd_Chrg", "Avg_Mdcr_Alowd_Amt", "Avg_Mdcr_Pymt_Amt"
]
partb = pd.read_csv(partb_file, dtype=str, usecols=partb_usecols)
print(f"Part B: {partb.shape[0]:,} rows, {partb.shape[1]} columns")

# Sanity check
if partb.shape[0] < 5_000_000:
    print(f"⚠️ WARNING: Only {partb.shape[0]:,} rows. National Part B by-service typically has 9M+ rows.")
else:
    print(f"✓ Row count looks reasonable for national Part B data")

In [ ]:
# Load NPPES — chunked, filter to MA on the fly
nppes_candidates = glob.glob("2_datasets/NPPES Data/npidata_pfile_*.csv")
# Exclude the fileheader file
nppes_candidates = [f for f in nppes_candidates if "fileheader" not in f]
if not nppes_candidates:
    raise FileNotFoundError(
        "NPPES file not found. Expected 'npidata_pfile_*.csv' in 2_datasets/NPPES Data/. "
        "Download from https://download.cms.gov/nppes/NPI_Files.html and extract the ZIP."
    )
nppes_file = nppes_candidates[0]
print(f"Using NPPES file: {nppes_file}")
print(f"File size: {os.path.getsize(nppes_file) / 1e9:.1f} GB")

# Key columns to keep (minimizes memory — full file has 330 columns)
nppes_cols = [
    "NPI",
    "Entity Type Code",
    "Provider Last Name (Legal Name)",
    "Provider First Name",
    "Provider Credential Text",
    "Provider Business Practice Location Address State Name",
    "Provider Business Practice Location Address City Name",
    "Provider Business Practice Location Address Postal Code",
    "Healthcare Provider Taxonomy Code_1",
    "Provider Enumeration Date",
    "NPI Deactivation Date",
]

# Chunked load, filter to MA
print("Loading NPPES and filtering to MA (this takes a few minutes)...")
ma_chunks = []
total_rows = 0
for chunk in pd.read_csv(nppes_file, dtype=str, usecols=nppes_cols, chunksize=50_000):
    total_rows += len(chunk)
    ma_chunk = chunk[
        chunk["Provider Business Practice Location Address State Name"]
        .str.upper().str.strip()
        .isin(["MA", "MASSACHUSETTS"])
    ]
    if len(ma_chunk) > 0:
        ma_chunks.append(ma_chunk)

nppes_ma = pd.concat(ma_chunks, ignore_index=True)
print(f"NPPES total rows processed: {total_rows:,}")
print(f"NPPES MA providers: {nppes_ma.shape[0]:,}")

# Sanity check: expect ~80k-120k MA NPIs
if nppes_ma.shape[0] < 30_000:
    print(f"⚠️ WARNING: Only {nppes_ma.shape[0]:,} MA NPIs. Expected ~80k-120k. Check filter column.")
elif nppes_ma.shape[0] > 200_000:
    print(f"⚠️ WARNING: {nppes_ma.shape[0]:,} MA NPIs is unusually high. Filter may not be working.")
else:
    print(f"✓ MA NPI count looks reasonable")

## Step 2 — Exploratory Data Analysis

**What this does:** Before cleaning or joining anything, understand what we're working with. Schema, shape, null rates, distributions, and key charts for each dataset.

In [ ]:
# --- QPP/MIPS EDA ---
print("=" * 60)
print("QPP/MIPS FINAL SCORES")
print("=" * 60)

print(f"\nShape: {qpp.shape}")
print(f"\nColumns: {list(qpp.columns)}")

# Note: most QPP columns have a leading space in the name
# Strip column names for easier access
qpp.columns = qpp.columns.str.strip()
print(f"\nColumns (cleaned): {list(qpp.columns)}")

print(f"\nNull rates:")
print((qpp.isnull().sum() / len(qpp) * 100).round(1).to_string())

print(f"\nData types:")
print(qpp.dtypes.to_string())

print(f"\nUnique NPIs: {qpp['NPI'].nunique():,}")
print(f"Source distribution:")
print(qpp["source"].value_counts().to_string())

In [ ]:
# QPP visualizations

# MIPS final score distribution
qpp_scores = pd.to_numeric(qpp["final_MIPS_score"], errors="coerce")
fig1 = px.histogram(
    qpp_scores.dropna(),
    nbins=50,
    title="QPP: MIPS Final Score Distribution (National)",
    labels={"value": "MIPS Final Score", "count": "Providers"},
)
fig1.show()

# Category score distributions
category_cols = {
    "Quality_category_score": "Quality",
    "PI_category_score": "Promoting Interoperability",
    "IA_category_score": "Improvement Activities",
    "Cost_category_score": "Cost",
}
for col, label in category_cols.items():
    vals = pd.to_numeric(qpp[col], errors="coerce").dropna()
    if len(vals) > 0:
        fig = px.histogram(vals, nbins=30, title=f"QPP: {label} Category Score Distribution")
        fig.show()
    print(f"{label}: {len(vals):,} non-null values, mean={vals.mean():.1f}, median={vals.median():.1f}")

# No low-volume flag column — will infer from Part B
print("\nNo explicit low-volume flag in QPP data. Will infer from Part B billing volume.")

In [ ]:
# --- Care Compare EDA ---
print("=" * 60)
print("CARE COMPARE — CLINICIAN DATA")
print("=" * 60)

print(f"\nShape: {care_compare.shape}")
print(f"\nColumns: {list(care_compare.columns)}")

print(f"\nNull rates:")
print((care_compare.isnull().sum() / len(care_compare) * 100).round(1).to_string())

print(f"\nData types:")
print(care_compare.dtypes.to_string())

print(f"\nUnique NPIs: {care_compare['NPI'].nunique():,}")

In [ ]:
# Care Compare visualizations

# Telehealth flag
fig = px.bar(
    care_compare["Telehlth"].value_counts().reset_index(),
    x="Telehlth", y="count",
    title="Care Compare: Telehealth Flag Distribution",
)
fig.show()

# Primary specialty distribution
fig = px.bar(
    care_compare["pri_spec"].value_counts().head(20).reset_index(),
    x="pri_spec", y="count",
    title="Care Compare: Top 20 Primary Specialties",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

# Distinct specialties per NPI (primary + secondary)
spec_cols = ["pri_spec", "sec_spec_1", "sec_spec_2", "sec_spec_3", "sec_spec_4"]
spec_count = care_compare[spec_cols].notna().sum(axis=1)
fig = px.histogram(
    spec_count,
    title="Care Compare: Number of Specialties per Provider",
    labels={"value": "Specialty Count", "count": "Providers"},
)
fig.show()

In [ ]:
# --- Part B EDA ---
print("=" * 60)
print("MEDICARE PART B UTILIZATION")
print("=" * 60)

print(f"\nShape: {partb.shape}")
print(f"\nColumns: {list(partb.columns)}")

print(f"\nNull rates:")
print((partb.isnull().sum() / len(partb) * 100).round(1).to_string())

print(f"\nData types:")
print(partb.dtypes.to_string())

print(f"\nUnique NPIs: {partb['Rndrng_NPI'].nunique():,}")

# Quick stats on numeric columns
for col in ["Tot_Benes", "Tot_Srvcs", "Avg_Sbmtd_Chrg"]:
    vals = pd.to_numeric(partb[col], errors="coerce")
    print(f"\n{col} stats:")
    print(vals.describe().to_string())

In [ ]:
# Part B visualizations

# Top specialties by claim lines
fig = px.bar(
    partb["Rndrng_Prvdr_Type"].value_counts().head(15).reset_index(),
    x="Rndrng_Prvdr_Type", y="count",
    title="Part B: Top 15 Specialties by Claim Lines",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

# Total charges per NPI (computed as Avg_Sbmtd_Chrg * Tot_Srvcs)
partb["_est_total_chrg"] = (
    pd.to_numeric(partb["Avg_Sbmtd_Chrg"], errors="coerce")
    * pd.to_numeric(partb["Tot_Srvcs"], errors="coerce")
)
npi_charges = partb.groupby("Rndrng_NPI")["_est_total_chrg"].sum().reset_index(name="est_total_charges")

fig = px.histogram(
    npi_charges["est_total_charges"].clip(upper=npi_charges["est_total_charges"].quantile(0.99)),
    nbins=50,
    title="Part B: Estimated Total Charges per NPI (clipped at 99th pctile)",
    labels={"value": "Estimated Total Charges ($)", "count": "Providers"},
)
fig.show()

print(f"Charge stats per NPI:")
print(npi_charges["est_total_charges"].describe().to_string())

# Clean up temp column
partb.drop(columns=["_est_total_chrg"], inplace=True)

In [ ]:
# --- NPPES (MA) EDA ---
print("=" * 60)
print("NPPES — MASSACHUSETTS PROVIDERS")
print("=" * 60)

print(f"\nShape: {nppes_ma.shape}")
print(f"\nColumns: {list(nppes_ma.columns)}")

print(f"\nNull rates:")
print((nppes_ma.isnull().sum() / len(nppes_ma) * 100).round(1).to_string())

print(f"\nEntity Type Code distribution:")
print(nppes_ma["Entity Type Code"].value_counts().to_string())

print(f"\nUnique NPIs: {nppes_ma['NPI'].nunique():,}")

# Check for deactivated NPIs
deactivated = nppes_ma["NPI Deactivation Date"].notna().sum()
print(f"\nDeactivated NPIs: {deactivated:,} ({deactivated/len(nppes_ma)*100:.1f}%)")

In [ ]:
# NPPES MA visualizations

# Provider type (Entity Type 1=Individual, 2=Organization)
fig = px.bar(
    nppes_ma["Entity Type Code"].value_counts().reset_index(),
    x="Entity Type Code", y="count",
    title="NPPES MA: Entity Type (1=Individual, 2=Organization)",
)
fig.show()

# Top taxonomy codes (proxy for specialty)
fig = px.bar(
    nppes_ma["Healthcare Provider Taxonomy Code_1"].value_counts().head(20).reset_index(),
    x="Healthcare Provider Taxonomy Code_1", y="count",
    title="NPPES MA: Top 20 Taxonomy Codes",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

# City distribution
fig = px.bar(
    nppes_ma["Provider Business Practice Location Address City Name"].value_counts().head(20).reset_index(),
    x="Provider Business Practice Location Address City Name", y="count",
    title="NPPES MA: Top 20 Cities",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

# ZIP code distribution
zip_counts = nppes_ma["Provider Business Practice Location Address Postal Code"].str[:5].value_counts().head(25)
fig = px.bar(
    zip_counts.reset_index(),
    x="Provider Business Practice Location Address Postal Code", y="count",
    title="NPPES MA: Top 25 ZIP Codes",
)
fig.update_layout(xaxis_tickangle=-45)
fig.show()

## Step 3 — Data Preprocessing & Cleaning

**What this does:** Harmonize NPIs, filter to MA, aggregate Part B to per-NPI, deduplicate, and build the master table via left joins.

In [ ]:
# --- Step 3: Data Preprocessing & Cleaning ---

# 1. NPI harmonization across all datasets
def clean_npi(df, npi_col="NPI"):
    """Strip whitespace, cast to string, validate 10-digit format."""
    df[npi_col] = df[npi_col].astype(str).str.strip()
    valid_mask = df[npi_col].str.match(r"^\d{10}$")
    invalid_count = (~valid_mask).sum()
    if invalid_count > 0:
        print(f"  ⚠️ {invalid_count:,} invalid NPIs removed (not 10 digits)")
        df = df[valid_mask].copy()
    return df

print("NPI harmonization:")

print(f"\nNPPES MA: {len(nppes_ma):,} rows before")
nppes_ma = clean_npi(nppes_ma)
print(f"  → {len(nppes_ma):,} rows after")

print(f"\nQPP: {len(qpp):,} rows before")
qpp = clean_npi(qpp, npi_col="NPI")
print(f"  → {len(qpp):,} rows after")

print(f"\nCare Compare: {len(care_compare):,} rows before")
care_compare = clean_npi(care_compare, npi_col="NPI")
print(f"  → {len(care_compare):,} rows after")

print(f"\nPart B: {len(partb):,} rows before")
partb = clean_npi(partb, npi_col="Rndrng_NPI")
print(f"  → {len(partb):,} rows after")

In [ ]:
# 3. QPP cleaning — parse scores and deduplicate

# Parse MIPS final score to numeric
qpp["mips_final_score"] = pd.to_numeric(qpp["final_MIPS_score"], errors="coerce")

# Parse category scores
qpp["quality_score"] = pd.to_numeric(qpp["Quality_category_score"], errors="coerce")
qpp["cost_score"] = pd.to_numeric(qpp["Cost_category_score"], errors="coerce")
qpp["improvement_activities_score"] = pd.to_numeric(qpp["IA_category_score"], errors="coerce")
qpp["promoting_interoperability_score"] = pd.to_numeric(qpp["PI_category_score"], errors="coerce")

# No low-volume flag in QPP data — will infer from Part B
qpp["low_volume_flag"] = None

print(f"MIPS final score stats:")
print(qpp["mips_final_score"].describe().to_string())

# Deduplication — keep highest MIPS score per NPI
# Trade-off: biases upward. A provider under two TINs may have different scores
# reflecting different practice contexts. For MVP we accept this and document it.
dupes = qpp["NPI"].duplicated().sum()
print(f"\nDuplicate NPIs in QPP: {dupes:,}")
if dupes > 0:
    qpp = qpp.sort_values("mips_final_score", ascending=False).drop_duplicates(subset="NPI", keep="first")
    print(f"After dedup (kept highest score): {len(qpp):,} rows")

In [ ]:
# 4. Part B aggregation — per-NPI summary
# Note: Avg_Sbmtd_Chrg is per-service-line average, not total
# Compute estimated totals: avg_charge * num_services

partb["_tot_srvcs"] = pd.to_numeric(partb["Tot_Srvcs"], errors="coerce")
partb["_tot_benes"] = pd.to_numeric(partb["Tot_Benes"], errors="coerce")
partb["_avg_sbmtd_chrg"] = pd.to_numeric(partb["Avg_Sbmtd_Chrg"], errors="coerce")
partb["_avg_mdcr_alowd"] = pd.to_numeric(partb["Avg_Mdcr_Alowd_Amt"], errors="coerce")
partb["_est_total_chrg"] = partb["_avg_sbmtd_chrg"] * partb["_tot_srvcs"]
partb["_est_total_alowd"] = partb["_avg_mdcr_alowd"] * partb["_tot_srvcs"]

partb_agg = partb.groupby("Rndrng_NPI").agg(
    total_services=("_tot_srvcs", "sum"),
    total_beneficiaries=("_tot_benes", "max"),  # max not sum: Tot_Benes is per-service-line, summing double-counts
    est_total_submitted_charges=("_est_total_chrg", "sum"),
    est_total_medicare_allowed=("_est_total_alowd", "sum"),
    distinct_hcpcs=("HCPCS_Cd", "nunique"),
    primary_specialty=("Rndrng_Prvdr_Type", lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else None),
).reset_index()

partb_agg = partb_agg.rename(columns={"Rndrng_NPI": "NPI"})

print(f"Part B aggregated: {len(partb_agg):,} unique NPIs")
print(f"\nPer-NPI stats:")
print(partb_agg[["total_services", "total_beneficiaries", "est_total_submitted_charges"]].describe().to_string())

In [ ]:
# 5-6. Care Compare dedup and 7. Build master table

# Care Compare deduplication
cc_dupes = care_compare["NPI"].duplicated().sum()
print(f"Duplicate NPIs in Care Compare: {cc_dupes:,}")

if cc_dupes > 0:
    # Keep first occurrence for most columns
    # Aggregate facility-related columns as semicolon-separated lists
    affil_cols = ["Facility Name", "org_pac_id"]
    agg_dict = {}
    for col in care_compare.columns:
        if col == "NPI":
            continue
        elif col in affil_cols:
            agg_dict[col] = lambda x: "; ".join(x.dropna().unique())
        else:
            agg_dict[col] = "first"
    care_compare_dedup = care_compare.groupby("NPI").agg(agg_dict).reset_index()
    print(f"After dedup (affiliations aggregated): {len(care_compare_dedup):,} rows")
else:
    care_compare_dedup = care_compare

# Select Care Compare columns for the master table
# These are enrichment for deferred Step 4 business logic
cc_join_cols = ["NPI", "pri_spec", "Telehlth", "Facility Name", "org_pac_id",
                "num_org_mem", "Cred", "Med_sch", "Grd_yr"]
cc_for_join = care_compare_dedup[[c for c in cc_join_cols if c in care_compare_dedup.columns]]

# Build master table — NPPES MA as left side
master = nppes_ma[["NPI", "Entity Type Code", "Provider Last Name (Legal Name)",
                    "Provider First Name", "Provider Credential Text",
                    "Provider Business Practice Location Address City Name",
                    "Provider Business Practice Location Address Postal Code",
                    "Healthcare Provider Taxonomy Code_1"]].copy()

print(f"\nMaster table start: {len(master):,} MA NPIs")

# Left join QPP
qpp_join_cols = ["NPI", "mips_final_score", "low_volume_flag",
                 "quality_score", "cost_score", "improvement_activities_score",
                 "promoting_interoperability_score", "source"]
master = master.merge(qpp[qpp_join_cols], on="NPI", how="left")
qpp_matched = master["mips_final_score"].notna().sum()
print(f"QPP matched: {qpp_matched:,} / {len(master):,} ({qpp_matched/len(master)*100:.1f}%)")

# Left join Part B
master = master.merge(partb_agg, on="NPI", how="left")
partb_matched = master["total_services"].notna().sum()
print(f"Part B matched: {partb_matched:,} / {len(master):,} ({partb_matched/len(master)*100:.1f}%)")

# Left join Care Compare
master = master.merge(cc_for_join, on="NPI", how="left")
cc_matched = master["pri_spec"].notna().sum()
print(f"Care Compare matched: {cc_matched:,} / {len(master):,} ({cc_matched/len(master)*100:.1f}%)")

# Validate: row count should match NPPES MA
assert len(master) == len(nppes_ma), (
    f"Master table has {len(master):,} rows but NPPES MA has {len(nppes_ma):,}. "
    "Left join should preserve all NPPES rows. Check for duplicate NPIs in join sources."
)
print(f"\n✓ Master table: {len(master):,} rows (matches NPPES MA count)")

# Column summary
print(f"\nColumn list ({len(master.columns)} total):")
print(list(master.columns))

print(f"\nNull rates:")
print((master.isnull().sum() / len(master) * 100).round(1).sort_values(ascending=False).to_string())

## Output Schema

One row per MA NPI. MIPS scores where available, Part B utilization summary, Care Compare enrichment. This table is the input for the business logic brainstorm (Step 4, to be designed).

In [ ]:
# --- Output: Master table summary ---
print("=" * 60)
print("MASTER TABLE — READY FOR BUSINESS LOGIC BRAINSTORM")
print("=" * 60)

print(f"\nShape: {master.shape}")
print(f"\nKey stats:")
has_mips = master["mips_final_score"].notna().sum()
no_mips = master["mips_final_score"].isna().sum()
print(f"  NPIs with MIPS score: {has_mips:,} ({has_mips/len(master)*100:.1f}%)")
print(f"  NPIs without MIPS score: {no_mips:,} ({no_mips/len(master)*100:.1f}%)")

if has_mips > 0:
    print(f"\n  MIPS score distribution (scored providers only):")
    print(f"  {master['mips_final_score'].dropna().describe().to_string()}")

has_partb = master["total_services"].notna().sum()
print(f"\n  NPIs with Part B data: {has_partb:,} ({has_partb/len(master)*100:.1f}%)")

has_cc = master["pri_spec"].notna().sum()
print(f"  NPIs with Care Compare data: {has_cc:,} ({has_cc/len(master)*100:.1f}%)")

# Save checkpoint for brainstorming
master.to_csv("2_datasets/step2_master_table_checkpoint.csv", index=False)
master.to_parquet("2_datasets/step2_master_table_checkpoint.parquet", index=False)
print(f"\n✓ Checkpoint saved to 2_datasets/step2_master_table_checkpoint.csv and .parquet")
print(f"  This table is the input for the three-situation classification brainstorm.")